## init

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set all fonts to Arial size 7
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 7,
    'axes.titlesize': 7,
    'axes.labelsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7
})

In [3]:
from autoadsorbate import Surface
from ase.io import read, write
from ase.visualize import view
import numpy as np
from ase import Atoms
import random
import math

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator


from cft import Manifold
from autoadsorbate.Particle import get_cube_surface_pts, grid_round_cube
from cft.mesh_utils import compute_outward_vertex_normals_quads

from ase.io import read, write
from ase.visualize import view
from autoadsorbate import Fragment
from autoadsorbate.Surf import attach_fragment
from ase.constraints import FixAtoms
import copy

torch-sim-atomistic not installed, defaulting to sequential optimization


In [4]:
from mace.calculators import mace_mp
clean_calc = mace_mp(model=
                '/mnt/c/Users/ef/Desktop/tmp/mace-mh-nl-pbe.model',
                # '/mnt/c/Users/ef/Desktop/tmp/mace-omat-0-medium.model',
                device='cpu',
                head ='matpes_r2scan'
                )

/home/ef/venvs/mace_env/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


/home/ef/venvs/mace_env/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head matpes_r2scan out of ['matpes_r2scan', 'mp_pbe_refit_add', 'spice_wB97M', 'oc20_usemppbe', 'omol', 'omat_pbe']


In [139]:
p = .1
c = 20
scale = [5,5,2]
bulk = read('./TiN.cif')


slab = bulk.copy()*scale
slab.cell[2][2] += c
slab.positions[:,2] += c/2
slab.arrays['fragments'] = np.array([0 for _ in slab])

#make some vacancies
_s = Surface(slab)
n_inds = [atom.index for atom in _s.atoms if atom.symbol =='N' and atom.index in _s.surf_inds]
n_vac = random.sample(n_inds, math.ceil(p * len(n_inds)))
slab = slab[[atom.index for atom in slab if atom.index not in n_vac]]

# al_z = slab.positions[np.where(slab.positions[:,2] == np.max(slab.positions[:,2]))[0][0]][2]
# al_pos = slab.cell[0]*0.5+slab.cell[1]*0.5 + [0,0,al_z]

# for x in range(3):
#     for y in range(3):
#         for z in range(1,3):
#             print(x,y,z)
#             slab += Atoms(['Al'], [al_pos+[x*2,y*2,z*2]])

slab.set_constraint(FixAtoms(indices=[atom.index for atom in slab if atom.position[2] < slab.cell[2][2]*.5]))
slab.rattle(stdev=.2)

view(slab)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

In [5]:
# from ase.optimize import BFGS

# slab.calc = clean_calc
# opt = BFGS(slab, trajectory='relax.xyz')
# opt.run(fmax=0.1)

In [23]:
# slab = read('/mnt/c/Users/ef/Desktop/tmp/elemynt/relax_run.xyz', index=-1)
# write('/mnt/c/Users/ef/Desktop/tmp/elemynt/relax_run_slab-1.xyz', slab)

# trj = read('/mnt/c/Users/ef/Desktop/tmp/elemynt/relax_run.xyz', index=':')
# write('/mnt/c/Users/ef/Desktop/tmp/elemynt/relax_run_slab_traj.xyz', trj)


# trj = read('/home/ef/liac22_home/git/CFT/examples/ald/md/md.xyz', index=':')
# write('/mnt/c/Users/ef/Desktop/tmp/elemynt/md_run_slab_traj_skip10.xyz', trj[::10])
trj = trj[::10]

In [ ]:
view(trj)

<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/ef/venvs/mace_env/lib/python3.12/site-packages/ase/gui/pipe.py", line 34, in <module>
    main()
  File "/home/ef/venvs/mace_env/lib/python3.12/site-packages/ase/gui/pipe.py", line 30, in main
    plt.show()
  File "/home/ef/venvs/mace_env/lib/python3.12/site-packages/matplotlib/pyplot.py", line 614, in show
    return _get_backend_mod().show(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ef/venvs/mace_env/lib/python3.12/site-packages/matplotlib_inline/backend_inline.py", line 90, in show
    display(
  File "/home/ef/venvs/mace_env/lib/python3.12/site-packages/IPython/core/display_functions.py", line 245, in display
    print(*objs)
ValueError: I/O operation on closed file.


## manifold

In [ ]:
from cft import Manifold

In [34]:
slab = trj[-1].copy()
write('slab_for_manifol.xyz', slab)

In [ ]:
m = Manifold(slab,
                precision=0.5,
                touch_sphere_size =2.5,
                wrap_on='sites',
                calc = clean_calc
                )

In [33]:
m.save_ply(filename='/mnt/c/Users/ef/Desktop/tmp/elemynt/manifold_clean.ply')

Saving grid to file: /mnt/c/Users/ef/Desktop/tmp/elemynt/manifold_clean.ply


In [29]:
m.view_grid(inclde_atoms=True)

In [ ]:
def make_custom_probe_scan(slab):
    f = Fragment('Cl[P+](C)(C)C', to_initialize=1)
    for atoms in f.conformers:
        for atom in atoms:
            if atom.symbol =='P':
                atom.symbol ='Al'

    m = Manifold(slab,
                precision=1,
                touch_sphere_size =2.5,
                wrap_on='atoms',
                calc = clean_calc)
    m.normals*=-1

    m.run_probe_scan(probes=[f, Fragment('ClC', to_initialize=1), (Atoms(['Al'], [0,0,0]))])
    m.write_grid('grd.xyz')


In [111]:
m.write_grid('grd.xyz')

In [36]:
comp_grid = read('/mnt/c/Users/ef/Desktop/tmp/elemynt/grd.xyz')

In [52]:
grd_alme2  = read('/mnt/c/Users/ef/Desktop/tmp/elemynt/AlMe2_grd.xyz')

grd_alme2.arrays.keys()

dict_keys(['numbers', 'positions', 'area', 'e_S1SP+1(C)C_0', 'grad_norm_e_S1SP+1(C)C_0', 'grad_e_S1SP+1(C)C_0'])

In [ ]:
f = Fragment('S1S[P+]1(C)C', to_initialize=1)
f.view()

User requested to_initialize = 1 conformers.
After pruning with 0.5; len(conformer_trj) = 1 unique conformers are found.


[15:07:13] UFFTYPER: Unrecognized charge state for atom: 1
[15:07:13] UFFTYPER: Unrecognized charge state for atom: 1


In [55]:
from cft.mesh_utils import values_to_colors

recepie_dict = {
    # 'AlMe3': 'e_ClP+(C)(C)C_0',
    # 'Al':'e_ClAl_0',
    # 'Me':'e_ClC_0',
    'AlMe2': 'e_S1SP+1(C)C_0'
    }

for name, key in recepie_dict.items():

    if name == 'AlMe2':
        vals = grd_alme2.arrays[key] + comp_grid.arrays['e_ClC_0'] - comp_grid.arrays['e_ClP+(C)(C)C_0']
    else:
        vals = comp_grid.arrays[key]
    
    vals-=vals[0]

    colors = [values_to_colors(v, [-5,5], reference_value=0, palette_nam="plasma") for v in vals]

    filename = f'/mnt/c/Users/ef/Desktop/tmp/elemynt/manifold_{name}.ply'
    m.save_ply(filename=filename, vertex_colors=colors)

Saving grid to file: /mnt/c/Users/ef/Desktop/tmp/elemynt/manifold_AlMe2.ply


## population

In [49]:
f = Fragment('Cl[P+](C)(C)C', to_initialize=100)
for atoms in f.conformers:
    for atom in atoms:
        if atom.symbol =='P':
            atom.symbol ='Al'

# m = Manifold(slab,
#                 precision=1,
#                 touch_sphere_size =2.5,
#                 wrap_on='atoms',
#                 calc = clean_calc)

# m.normals*=-1


pop_traj = []

for coverage in [.8]:
# for coverage in [.2,.5,.6,.7,.8,.9,1.]:
    m.make_fragment_population(
                population_size = 1,
                fragment = f,
                coverage = coverage
                )
    pop_traj+=m.surf_population

view(pop_traj)

User requested to_initialize = 100 conformers.
After pruning with 0.5; len(conformer_trj) = 1 unique conformers are found.


[15:29:00] UFFTYPER: Unrecognized charge state for atom: 1
[15:29:00] UFFTYPER: Unrecognized charge state for atom: 1


<Popen: returncode: None args: ['/home/ef/venvs/mace_env/bin/python', '-m', ...>

In [166]:
for a in pop_traj:
    a.arrays['fragments'] += (a.arrays['fragments'] > 0) * 10

In [50]:
write('/mnt/c/Users/ef/Desktop/tmp/elemynt/pop_traj.xyz', pop_traj)

In [1]:
# pop_traj = read('/mnt/c/Users/ef/Desktop/tmp/pop_traj.xyz', index = ':')
# for i, atoms in enumerate(pop_traj):
#     atoms.set_constraint(FixAtoms(indices=[atom.index for atom in atoms if atom.symbol in ['N', 'Ti']]))
#     atoms.calc = clean_calc
#     opt = BFGS(atoms, trajectory='tmp.xyz')
#     print(f'{i = }')
#     opt.run(fmax=0.5)

In [153]:
m.surf_population[0].arrays.keys()

dict_keys(['numbers', 'positions', 'spacegroup_kinds', 'fragments'])

In [ ]:
for atoms in m.surf_population:


m.evaluate_surf_population()

Calculating energies: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]
